# Diagnóstico de calidad de los datos de Instacart

**Actividad:** GA1-220501094-01-AA1-EV06 – Diagnosticar la calidad de datos: preparación, exploración y documentación
**Rol:** analista de datos responsable de evaluar la calidad inicial de los datasets del proyecto (desarrollo individual)
**Aprendiz:** Juan Camilo Sarrazola Agudelo

> **Cómo está construido este notebook:** igual que en la EV04, las cifras de las interpretaciones no están escritas a mano.
> Se calculan con código a partir de los DataFrames cargados, de modo que el diagnóstico siempre corresponde a los datos.
> En esta actividad **no se limpia ni se modifica ningún registro**: solo se diagnostica, se interpreta y se documenta.

---
## Momento 1. Reconocimiento del proyecto y de los datasets

### 1.1 Nombre y propósito del proyecto
**Proyecto:** *Explorando Instacart: descubre patrones de compra.*

**Propósito:** describir los hábitos de compra de los clientes de Instacart (plataforma de compras de supermercado) a partir de
sus pedidos: cuándo compran, cada cuánto vuelven a comprar y qué productos repiten, para que el equipo de operaciones
pueda tomar decisiones con información confiable.

### 1.2 Pregunta analítica (recuperada de la EV04)
> **¿Qué hábitos de compra presentan los clientes de Instacart —cuándo compran, cada cuánto vuelven a comprar y qué productos recompran—?**

Se concreta en las tres preguntas de análisis formuladas en la EV04:
1. ¿Cada cuántos días, en promedio, vuelven a comprar los clientes? (`days_since_prior_order`)
2. ¿Qué proporción de los artículos de los pedidos son recompras? (`reordered`)
3. ¿En qué días y horas se concentran los pedidos de los clientes recurrentes? (`order_dow`, `order_hour_of_day`, `order_number`)

### 1.3 Datasets del proyecto y su relación con el problema

| N.º | Archivo | DataFrame | Qué contiene | Relación con la pregunta analítica |
|---|---|---|---|---|
| 1 | `instacart_orders.csv` | `orders` | Un registro por pedido: usuario, número de pedido, día, hora y días desde el pedido anterior. | Responde *cuándo* compran y *cada cuánto* vuelven. |
| 2 | `products.csv` | `products` | Catálogo: identificador, nombre, pasillo y departamento del producto. | Permite saber *qué* productos se compran. |
| 3 | `aisles.csv` | `aisles` | Tabla de consulta `aisle_id` → nombre del pasillo. | Agrupa productos por pasillo. |
| 4 | `departments.csv` | `departments` | Tabla de consulta `department_id` → nombre del departamento. | Agrupa productos por departamento. |
| 5 | `order_products.csv` | `order_products` | Detalle de pedidos: un registro por producto, posición en el carrito y si es recompra. | Responde *qué* se recompra y une pedidos con productos. |

### 1.4 Aspectos de calidad que se deben revisar antes de usar los datos
- **Completitud:** columnas con valores ausentes y si la ausencia tiene una explicación lógica.
- **Tipos de datos:** que cada variable tenga un tipo coherente con lo que representa (enteros, códigos, texto, binarios).
- **Unicidad:** filas completamente duplicadas y claves (`order_id`, `product_id`) repetidas.
- **Validez de rangos:** horas entre 0 y 23, días de la semana entre 0 y 6, `reordered` solo 0 o 1, etc.
- **Consistencia lógica:** que las variables no se contradigan entre sí (por ejemplo, primer pedido sin pedido anterior).
- **Integridad referencial:** que las claves de una tabla existan en la tabla con la que se relacionan.
- **Limitaciones:** información que falta para responder la pregunta analítica.

---
## Momento 2. Preparación del entorno y carga de datos

### 2.1 Librerías y funciones auxiliares

In [ ]:
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 90)


def md(texto):
    # Muestra un texto en formato Markdown dentro del notebook.
    display(Markdown(texto))


def fmt(n):
    # Entero con punto de miles (estilo es-CO): 478967 -> 478.967
    return f"{n:,.0f}".replace(",", ".")


def fmt2(x):
    # Decimal con coma decimal y punto de miles: 11.1017 -> 11,10
    return f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def pct(parte, total):
    # Porcentaje con dos decimales en formato es-CO.
    return fmt2(100 * parte / total)


print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)

### 2.2 Ubicación de los archivos
Los CSV no se versionan en GitHub (ver `README.md`). El notebook los busca primero en `data/raw/` (estructura de referencia de la guía)
y luego en `data/` (estructura usada desde la EV04). Si trabaja en Google Colab, agregue su ruta de Drive a `CANDIDATAS`.

In [ ]:
ARCHIVOS = {
    "orders": "instacart_orders.csv",
    "products": "products.csv",
    "aisles": "aisles.csv",
    "departments": "departments.csv",
    "order_products": "order_products.csv",
}

CANDIDATAS = [Path("../data/raw"), Path("../data"), Path("data/raw"), Path("data")]


def encontrar_ruta_datos():
    # Devuelve la primera carpeta que contiene los cinco CSV del proyecto.
    for carpeta in CANDIDATAS:
        if all((carpeta / archivo).exists() for archivo in ARCHIVOS.values()):
            return carpeta
    raise FileNotFoundError(
        "No se encontraron los cinco CSV de Instacart. Cópielos en data/raw/ o data/ (ver README.md)."
    )


RUTA_DATOS = encontrar_ruta_datos()
print("Carpeta de datos:", RUTA_DATOS.as_posix())

### 2.3 Carga de cada dataset en un DataFrame independiente
Los cinco archivos son texto delimitado que usa **punto y coma (`;`)** como separador, por eso se cargan con `pd.read_csv(..., sep=";")`.

In [ ]:
orders = pd.read_csv(RUTA_DATOS / ARCHIVOS["orders"], sep=";")
products = pd.read_csv(RUTA_DATOS / ARCHIVOS["products"], sep=";")
aisles = pd.read_csv(RUTA_DATOS / ARCHIVOS["aisles"], sep=";")
departments = pd.read_csv(RUTA_DATOS / ARCHIVOS["departments"], sep=";")
order_products = pd.read_csv(RUTA_DATOS / ARCHIVOS["order_products"], sep=";")

# Diccionario para recorrer los cinco DataFrames con un solo bucle
dataframes = {
    "orders": orders,
    "products": products,
    "aisles": aisles,
    "departments": departments,
    "order_products": order_products,
}

### 2.4 Verificación de la carga
Se comprueba que cada archivo produjo un DataFrame con más de una columna (si el separador fuera incorrecto, todo quedaría en **una sola columna**)
y con al menos un registro.

In [ ]:
verificacion = pd.DataFrame(
    [
        {
            "DataFrame": nombre,
            "archivo": ARCHIVOS[nombre],
            "tamaño archivo (MB)": round((RUTA_DATOS / ARCHIVOS[nombre]).stat().st_size / 1e6, 2),
            "filas": df.shape[0],
            "columnas": df.shape[1],
            "memoria en pandas (MB)": round(df.memory_usage(deep=True).sum() / 1e6, 1),
            "carga correcta": df.shape[0] > 0 and df.shape[1] > 1,
        }
        for nombre, df in dataframes.items()
    ]
)
assert verificacion["carga correcta"].all(), "Algún archivo no se cargó correctamente: revise el separador."
display(verificacion)
md(f"Los **{len(dataframes)}** datasets se cargaron correctamente y suman **{fmt(verificacion['filas'].sum())}** registros.")

---
## Momento 3. Inspección de la estructura de los datasets

Para cada DataFrame se ejecutan los ocho comandos de inspección que pide la actividad:

| Comando | Propósito |
|---|---|
| `head()` | Visualizar los primeros registros. |
| `tail()` | Visualizar los últimos registros. |
| `shape` | Identificar el número de filas y columnas. |
| `columns` | Consultar los nombres de las variables. |
| `dtypes` | Reconocer los tipos de datos. |
| `info()` | Examinar la estructura y los valores no nulos. |
| `describe()` | Estadísticas descriptivas de las variables numéricas. |
| `describe(include='object')` | Describir las variables de texto, cuando existan. |

> **Nota sobre pandas 3:** desde pandas 3.0 las columnas de texto se cargan con el tipo **`str`** y ya no como `object`.
> `describe(include='object')` las sigue incluyendo por compatibilidad (pandas emite un aviso de obsolescencia, que aquí se silencia).
> Por eso, en `dtypes` las columnas de texto aparecen como `str`.

In [ ]:
def tiene_texto(df):
    # True si el DataFrame tiene al menos una columna de texto (object o str).
    return any(pd.api.types.is_string_dtype(df[col]) or df[col].dtype == object for col in df.columns)


def inspeccionar(nombre, df):
    # Ejecuta los ocho comandos de inspección estructural sobre un DataFrame.
    md(f"### Inspección de `{nombre}`")

    md("**`head()`** · primeros 5 registros")
    display(df.head())

    md("**`tail()`** · últimos 5 registros")
    display(df.tail())

    md("**`shape`** · (filas, columnas)")
    print(df.shape)

    md("**`columns`** · nombres de las variables")
    print(list(df.columns))

    md("**`dtypes`** · tipo de dato de cada columna")
    print(df.dtypes)

    md("**`info()`** · estructura y valores no nulos")
    df.info()

    md("**`describe()`** · variables numéricas")
    display(df.describe())

    md("**`describe(include='object')`** · variables de texto")
    if tiene_texto(df):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            display(df.describe(include="object"))
    else:
        print(f"{nombre} no tiene columnas de texto: este resumen no aplica.")

### 3.1 `orders` (pedidos)

In [ ]:
inspeccionar("orders", orders)

In [ ]:
desc_o = orders.describe()
md(f'''
**Características principales de `orders`:**
- {fmt(orders.shape[0])} pedidos y {orders.shape[1]} columnas, todas numéricas. Los pedidos pertenecen a **{fmt(orders["user_id"].nunique())}** usuarios distintos.
- `order_hour_of_day` va de {desc_o.loc["min", "order_hour_of_day"]:.0f} a {desc_o.loc["max", "order_hour_of_day"]:.0f} y `order_dow` de {desc_o.loc["min", "order_dow"]:.0f} a {desc_o.loc["max", "order_dow"]:.0f}: códigos de hora y de día de la semana.
- `order_number` va de {desc_o.loc["min", "order_number"]:.0f} a {desc_o.loc["max", "order_number"]:.0f} (mediana {desc_o.loc["50%", "order_number"]:.0f}).
- `days_since_prior_order` es la única columna `float64`: tiene `count` = {fmt(desc_o.loc["count", "days_since_prior_order"])} frente a {fmt(len(orders))} filas, y va de {desc_o.loc["min", "days_since_prior_order"]:.0f} a {desc_o.loc["max", "days_since_prior_order"]:.0f} días.
- `orders` no tiene columnas de texto, por eso `describe(include='object')` no aplica.
''')

### 3.2 `products` (catálogo de productos)

In [ ]:
inspeccionar("products", products)

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    desc_p = products.describe(include="object")
md(f'''
**Características principales de `products`:**
- {fmt(products.shape[0])} productos y {products.shape[1]} columnas: tres identificadores enteros y el nombre (`{products["product_name"].dtype}`).
- `product_name` tiene `count` = {fmt(desc_p.loc["count", "product_name"])}: faltan **{fmt(products["product_name"].isna().sum())}** nombres.
- `unique` = {fmt(desc_p.loc["unique", "product_name"])} y `freq` = {desc_p.loc["freq", "product_name"]}: ningún nombre se repite *exactamente*, aunque más adelante se revisa si se repite con otras mayúsculas.
- `aisle_id` va de {products["aisle_id"].min()} a {products["aisle_id"].max()} y `department_id` de {products["department_id"].min()} a {products["department_id"].max()}, coherente con los catálogos de pasillos y departamentos.
''')

### 3.3 `aisles` (pasillos)

In [ ]:
inspeccionar("aisles", aisles)

In [ ]:
md(f'''
**Características principales de `aisles`:**
- Tabla de consulta pequeña: {fmt(aisles.shape[0])} pasillos y {aisles.shape[1]} columnas (`aisle_id` entero y `aisle` texto).
- Los {fmt(aisles["aisle"].nunique())} nombres son únicos y no hay ausentes. Sin embargo, uno de los pasillos se llama **`{aisles.loc[aisles["aisle"].str.lower().eq("missing"), "aisle"].iloc[0]}`** (id {aisles.loc[aisles["aisle"].str.lower().eq("missing"), "aisle_id"].iloc[0]}): es una categoría de relleno, no un pasillo real.
''')

### 3.4 `departments` (departamentos)

In [ ]:
inspeccionar("departments", departments)

In [ ]:
md(f'''
**Características principales de `departments`:**
- {fmt(departments.shape[0])} departamentos y {departments.shape[1]} columnas, sin ausentes ni nombres repetidos.
- Igual que en `aisles`, existe un departamento llamado **`missing`** (id {departments.loc[departments["department"].eq("missing"), "department_id"].iloc[0]}) y otro llamado `other`: categorías genéricas que no describen el producto.
''')

### 3.5 `order_products` (detalle de los pedidos)

In [ ]:
inspeccionar("order_products", order_products)

In [ ]:
desc_op = order_products.describe()
md(f'''
**Características principales de `order_products`:**
- Es el dataset más grande: **{fmt(order_products.shape[0])}** registros y {order_products.shape[1]} columnas numéricas (≈{fmt(order_products.memory_usage(deep=True).sum() / 1e6)} MB en memoria).
- Cada fila es *un producto dentro de un pedido*; abarca {fmt(order_products["order_id"].nunique())} pedidos y {fmt(order_products["product_id"].nunique())} productos distintos.
- `add_to_cart_order` es `float64`, con `count` = {fmt(desc_op.loc["count", "add_to_cart_order"])} (faltan {fmt(order_products["add_to_cart_order"].isna().sum())}) y máximo {desc_op.loc["max", "add_to_cart_order"]:.0f}.
- `reordered` tiene mínimo {desc_op.loc["min", "reordered"]:.0f}, máximo {desc_op.loc["max", "reordered"]:.0f} y media {fmt2(desc_op.loc["mean", "reordered"])}: es una variable binaria; el {pct(order_products["reordered"].sum(), len(order_products))} % de los artículos son recompras.
''')

### 3.6 Tabla de diagnóstico estructural
Resumen de la inspección: filas, columnas, tipos de datos y observaciones de cada dataset. La tabla también se guarda en
`reports/diagnostico_estructural.csv` para que el informe técnico use exactamente los mismos resultados.

In [ ]:
def resumir_tipos(df):
    # Cuenta cuántas columnas hay de cada tipo, por ejemplo: 'int64 ×5, float64 ×1'.
    conteo = df.dtypes.astype(str).value_counts()
    return ", ".join(f"{tipo} ×{n}" for tipo, n in conteo.items())


def observar(nombre, df):
    # Observaciones automáticas de la inspección estructural.
    notas = []
    nulos = df.isna().sum()
    for col, n in nulos[nulos > 0].items():
        notas.append(f"{col}: {fmt(n)} ausentes ({pct(n, len(df))} %)")
    for col in df.select_dtypes(include="float").columns:
        if (df[col].dropna() % 1 == 0).all():
            notas.append(f"{col} guarda enteros como float64")
    duplicadas = df.duplicated().sum()
    if duplicadas:
        notas.append(f"{fmt(duplicadas)} filas duplicadas")
    if not notas:
        notas.append("Sin ausentes ni filas duplicadas")
    return "; ".join(notas) + "."


diagnostico_estructural = pd.DataFrame(
    [
        {
            "Dataset": nombre,
            "Archivo": ARCHIVOS[nombre],
            "Filas": df.shape[0],
            "Columnas": df.shape[1],
            "Tipos de datos": resumir_tipos(df),
            "Observaciones": observar(nombre, df),
        }
        for nombre, df in dataframes.items()
    ]
)

RUTA_REPORTES = RUTA_DATOS.parent / "reports" if RUTA_DATOS.name == "data" else RUTA_DATOS.parent.parent / "reports"
RUTA_REPORTES.mkdir(exist_ok=True)
diagnostico_estructural.to_csv(RUTA_REPORTES / "diagnostico_estructural.csv", index=False, encoding="utf-8")

tabla = "| Dataset | Filas | Columnas | Tipos de datos | Observaciones |\n|---|---|---|---|---|\n"
tabla += "\n".join(
    f"| `{r.Dataset}` | {fmt(r.Filas)} | {r.Columnas} | {r[4]} | {r.Observaciones} |"
    for r in diagnostico_estructural.itertuples(index=False)
)
md(tabla)